In [ ]:
import json

with open("RESULTS/training_50_EneSou_BodOfWat_Org_PhyPhe_Loc_PhyArt_NatDis_Che_BodPar_MatExp_GeoFea_Org_IntArt_Sys_Ass_Met_FieOfStu_MetPhe_TimPer_Eco_Pol_NatPhe_Qua_Per_Dis_MeaDev_Sat.json") as f:
    json_file = json.load(f)
    
with open("RESULTS/golden_50_EneSou_BodOfWat_Org_PhyPhe_Loc_PhyArt_NatDis_Che_BodPar_MatExp_GeoFea_Org_IntArt_Sys_Ass_Met_FieOfStu_MetPhe_TimPer_Eco_Pol_NatPhe_Qua_Per_Dis_MeaDev_Sat.json") as f:
    json_file_g = json.load(f)

In [ ]:
print(len(json_file))
print(len(json_file_g))
print(len(json_file) + len(json_file_g))

In [ ]:
# 3. Load Data
from datasets import load_dataset
dataset_id = "P0L3/CliReNER_v_1_1_28_SILVER"

print(f"Loading dataset: {dataset_id}")
dataset = load_dataset(dataset_id)

# Extract labels (Assumes standard NER format)
# labels = dataset["train"].features["ner_tags"][0].names
labels = dataset["train"].features["ner_tags"].feature.names


In [ ]:
total = 0
for key in dataset:
    print(len(dataset[key]))
    total += len(dataset[key])
    
print(total)

In [ ]:
import seaborn as sns

sns.color_palette("Oranges")
print(sns.color_palette("Oranges").as_hex())

In [ ]:
from dataset_processing import cwed4eta_process_json_file, transform_to_ner_format, CLIRENER_LABELS_V1, ner_dataset_to_hf_format
import argparse
import pathlib


In [ ]:
data = cwed4eta_process_json_file("DATA/LABEL_STUDIO/project-6-at-2025-03-31-11-50-c08f75da.json")


In [ ]:
data

In [ ]:
from dataset_processing import BIODIVNER_DIR, load_biodivner, transform_to_ner_format, ner_dataset_to_hf_format, generate_consistent_label_map, BIODIVNER_LABELS, biodivner_process_bio_documents

In [ ]:
print("Converting from BIO tags to char spans.")
split_name = "train"
hf_name, safe_hf_name, output_dir = "BioDivNER", "BioDivNER", "PLOTS"


biodivner_structured_data = biodivner_process_bio_documents(
    data_dir=BIODIVNER_DIR, 
    labels_to_keep=BIODIVNER_LABELS, 
    split=[split_name]#, "validation", "test"]
    )

In [ ]:
transformed_dataset, tag_map = transform_to_ner_format(biodivner_structured_data, BIODIVNER_LABELS)

In [ ]:
dataset_train = ner_dataset_to_hf_format(transformed_dataset, tag_map, test_size=0, val_size=0)

In [ ]:
from EXPERIMENTS.calculate_hf_dataset_stats import process_dataset_split, plot_class_distribution

In [ ]:
stats, counts = process_dataset_split(dataset_train["train"], list(generate_consistent_label_map(BIODIVNER_LABELS).keys()))

In [ ]:
# Print to Console
print(f"{split_name:<12} | "
      f"{stats['Sentence Length (Mean)']:<10.2f} | "
      f"{stats['Entity Density (Avg entities/sent)']:<10.2f} | "
      f"{stats['Ratio (Median Len / Density)']:<10.2f} | "
      f"{stats['Ratio (Mean Len / Density)']:<10.2f} | "
      f"{stats['Span Length (Avg tokens/entity)']:<10.2f} | "
      f"{int(stats['Total Entities']):<10}")

# Generate Plot
plot_class_distribution(counts, split_name, hf_name, safe_hf_name, output_dir)

['[CLS]', 'the', 'increase', 'in', 'greenhouse', 'gas', 'emissions', 'has', 'significantly', 'affected', 'the', '[MASK]', 'balance', 'of', 'the', 'earth', '.', '[SEP]']


In [20]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, pipeline
import torch

# Load the pretrained model and tokenizer
model_list = ["P0L3/clirebert_clirevocab_uncased", "bert-base-uncased", "allenai/scibert_scivocab_uncased"]

for model_name in model_list:
    # model_name = "P0L3/clirebert_clirevocab_uncased"# "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # model = AutoModelForMaskedLM.from_pretrained(model_name)

    # # Move model to GPU if available
    # device = 0 if torch.cuda.is_available() else -1

    # # Create a fill-mask pipeline
    # fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer, device=device)

    # Example input from scientific climate literature
    text = "Sequestering carbon in soil is the most scalable and least costly option for CO2 removal in the coming decades [74]."
    # Get token IDs
    token_ids = tokenizer.encode(text, add_special_tokens=False)

    # Convert IDs to token strings
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    print(tokens)
# # Run prediction
# predictions = fill_mask(text)

# # Show top predictions
# print(text)
# print(10*">")
# for p in predictions:
#     print(f"{p['sequence']} — {p['score']:.4f}")


['sequester', '##ing', 'carbon', 'in', 'soil', 'is', 'the', 'most', 'scalable', 'and', 'least', 'costly', 'option', 'for', 'co2', 'removal', 'in', 'the', 'coming', 'decades', '[', '74', '].']
['se', '##quest', '##ering', 'carbon', 'in', 'soil', 'is', 'the', 'most', 'scala', '##ble', 'and', 'least', 'costly', 'option', 'for', 'co', '##2', 'removal', 'in', 'the', 'coming', 'decades', '[', '74', ']', '.']
['seques', '##tering', 'carbon', 'in', 'soil', 'is', 'the', 'most', 'scalable', 'and', 'least', 'costly', 'option', 'for', 'co', '##2', 'removal', 'in', 'the', 'coming', 'decades', '[', '74', ']', '.']


In [9]:
predictions

[{'score': 0.20367968082427979,
  'token': 11520,
  'token_str': 'cooling',
  'sequence': 'between 1981 and 2011, long - term hydrographic observations show warming ( by 0. 6 °c ) and cooling ( by 0. 06 ) of circumpolar deep water ( cdw ; ~ 200 – 600 m ) while the ocean surface cooled ( by 0. 2 °c ) and freshened ( by 0. 08 ).'},
 {'score': 0.0667988732457161,
  'token': 13845,
  'token_str': 'lowering',
  'sequence': 'between 1981 and 2011, long - term hydrographic observations show warming ( by 0. 6 °c ) and lowering ( by 0. 06 ) of circumpolar deep water ( cdw ; ~ 200 – 600 m ) while the ocean surface cooled ( by 0. 2 °c ) and freshened ( by 0. 08 ).'},
 {'score': 0.06316535919904709,
  'token': 13721,
  'token_str': 'melting',
  'sequence': 'between 1981 and 2011, long - term hydrographic observations show warming ( by 0. 6 °c ) and melting ( by 0. 06 ) of circumpolar deep water ( cdw ; ~ 200 – 600 m ) while the ocean surface cooled ( by 0. 2 °c ) and freshened ( by 0. 08 ).'},
 {'

## NER Examples

In [ ]:
import datasets
from EXPERIMENTS.evaluate_gold import load_and_merge_gold_data

dataset = load_and_merge_gold_data("P0L3/CliReNER_v_1_1_28_GOLD")


print(dataset[1])
print(dataset[0]["test"])
print(dataset[0]["test"][0]["ner_tags"])
print(dataset[0]["test"][0]["tokens"])

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192
['O', 'B-Asset', 'I-Asset', 'B-Body Part', 'I-Body Part', 'B-Body of Water', 'I-Body of Water', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease', 'B-Ecosystem', 'I-Ecosystem', 'B-Energy Source', 'I-Energy Source', 'B-Field of Study', 'I-Field of Study', 'B-Geographical Feature', 'I-Geographical Feature', 'B-Intellectual Artefact', 'I-Intellectual Artefact', 'B-Location', 'I-Location', 'B-Mathematical Expression', 'I-Mathematical Expression', 'B-Measuring Device', 'I-Measuring Device', 'B-Meteorological Phenomenon', 'I-Meteorological Phenomenon', 'B-Method', 'I-Method', 'B-Natural Disaster', 'I-Natural Disaster', 'B-Natural Phenomenon', 'I-Natural Phenomenon', 'B-Organism', 'I-Organism', 'B-Organization', 'I-Organization', 'B-Other', 'I-Other', 'B-Person', 'I-Person', 'B-Physical Artefact', 'I-Physical Artefact', 'B-Physical Phenomenon', 'I-

In [22]:
print(dataset[1])

['O', 'B-Asset', 'I-Asset', 'B-Body Part', 'I-Body Part', 'B-Body of Water', 'I-Body of Water', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease', 'B-Ecosystem', 'I-Ecosystem', 'B-Energy Source', 'I-Energy Source', 'B-Field of Study', 'I-Field of Study', 'B-Geographical Feature', 'I-Geographical Feature', 'B-Intellectual Artefact', 'I-Intellectual Artefact', 'B-Location', 'I-Location', 'B-Mathematical Expression', 'I-Mathematical Expression', 'B-Measuring Device', 'I-Measuring Device', 'B-Meteorological Phenomenon', 'I-Meteorological Phenomenon', 'B-Method', 'I-Method', 'B-Natural Disaster', 'I-Natural Disaster', 'B-Natural Phenomenon', 'I-Natural Phenomenon', 'B-Organism', 'I-Organism', 'B-Organization', 'I-Organization', 'B-Other', 'I-Other', 'B-Person', 'I-Person', 'B-Physical Artefact', 'I-Physical Artefact', 'B-Physical Phenomenon', 'I-Physical Phenomenon', 'B-Policy', 'I-Policy', 'B-Quantity', 'I-Quantity', 'B-Satellite', 'I-Satellite', 'B-System', 'I-System', 'B-Time Period',

In [130]:
import re
from collections import defaultdict

dataset[0]["test"] = dataset[0]["test"].shuffle()

# 1. Get the tag names from the dataset features
tags = dataset[1]

# 2. Helper to convert 'Body of Water' to 'BodyOfWater' for your LaTeX macros
def to_camel_case(label_name):
    return "".join(word.capitalize() for word in label_name.split())

# 3. Simple detokenizer to fix spaces around punctuation
def clean_text(text):
    text = re.sub(r' ([.,?!:;)])', r'\1', text)
    text = re.sub(r'(\() ', r'\1', text)
    return text

examples_per_type = defaultdict(list)
MAX_EXAMPLES = 2 # Change this to 3 if you want 3 examples per type
CONTEXT_WINDOW = 7 # Number of words to show before and after the entity

# 4. Extract examples
for row in dataset[0]["test"]:
    tokens = row['tokens']
    ner_ids = row['ner_tags']
    # print(tokens)
    # print(ner_ids)
    
    i = 0
    while i < len(tokens):
        tag_name = tags[ner_ids[i]]
        # print(tag_name)
        
        if tag_name.startswith("B-"):
            ent_type = tag_name[2:]
            
            # Skip if we already have enough examples for this category
            if len(examples_per_type[ent_type]) >= MAX_EXAMPLES:
                i += 1
                continue
                
            # Find the full entity span
            start_idx = i
            ent_tokens = [tokens[i]]
            i += 1
            while i < len(tokens) and tags[ner_ids[i]] == f"I-{ent_type}":
                ent_tokens.append(tokens[i])
                i += 1
            end_idx = i
            
            # Format the entity in LaTeX
            ent_text = " ".join(ent_tokens)
            camel_type = to_camel_case(ent_type)
            latex_entity = f"\\labbox{{{camel_type}}}{{{ent_text}}}"
            
            # Grab context window
            prefix = tokens[max(0, start_idx - CONTEXT_WINDOW) : start_idx]
            suffix = tokens[end_idx : min(len(tokens), end_idx + CONTEXT_WINDOW)]
            
            # Build the snippet
            snippet_tokens = []
            if start_idx > 0: snippet_tokens.append("...")
            snippet_tokens.extend(prefix)
            snippet_tokens.append(latex_entity)
            snippet_tokens.extend(suffix)
            if end_idx < len(tokens): snippet_tokens.append("...")
            
            snippet_str = clean_text(" ".join(snippet_tokens))
            
            # Avoid duplicate snippets
            if snippet_str not in examples_per_type[ent_type]:
                examples_per_type[ent_type].append(snippet_str)
        else:
            i += 1

# 5. Print out the LaTeX ready code
print("% --- COPY AND PASTE THIS INTO YOUR LATEX PRESENTATION ---")
print("\\begin{frame}[allowframebreaks]{NER Examples from Dataset}")
print("\\footnotesize")

# Sort alphabetically by entity type
for ent_type in sorted(examples_per_type.keys()):
    sentences = examples_per_type[ent_type]
    print(f"\\textbf{{{ent_type}}}")
    print("\\begin{itemize}")
    for sent in sentences:
        # Escape LaTeX special characters if any exist in the raw text (like %, &, _)
        sent = sent.replace("%", "\\%").replace("&", "\\&").replace("_", "\\_")
        print(f"    \\item {sent}")
    print("\\end{itemize}\n")
    
print("\\end{frame}")

% --- COPY AND PASTE THIS INTO YOUR LATEX PRESENTATION ---
\begin{frame}[allowframebreaks]{NER Examples from Dataset}
\footnotesize
\textbf{Asset}
\begin{itemize}
    \item ... In a study on \labbox{Asset}{iron} and steel scrap in the United Kingdom...
    \item ... In a study on iron and \labbox{Asset}{steel} scrap in the United Kingdom (UK...
\end{itemize}

\textbf{Body Part}
\begin{itemize}
    \item ... such as ALT is also high in \labbox{BodyPart}{cardiac} and skeletal muscle in CSL, creatine...
    \item ... ALT is also high in cardiac and \labbox{BodyPart}{skeletal} muscle in CSL, creatine kinase (...
\end{itemize}

\textbf{Body of Water}
\begin{itemize}
    \item ... and lower equatorial SSTs in the eastern \labbox{BodyOfWater}{Pacific}, resulting in the enhancement of the...
    \item ... anomalous positive sea level pressure in the \labbox{BodyOfWater}{North Pacific} extending to North America (Meehl et...
\end{itemize}

\textbf{Chemical}
\begin{itemize}
    \item ... Excludi

## Dataset loading

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'ner_tags'],
        num_rows: 150
    })
    validation: Dataset({
        features: ['id', 'text', 'tokens', 'ner_tags'],
        num_rows: 21
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'ner_tags'],
        num_rows: 21
    })
})

In [24]:
# from datasets import load_dataset
from dataset_processing import hf_dataset_to_gliner_format
from EXPERIMENTS.evaluate_gold import load_and_merge_gold_data
# Load the gold-standard evaluation set
print("Loading and merging from HF:")
dataset = load_and_merge_gold_data("P0L3/CliReNER_v_1_1_28_GOLD")

print("Dataset:", dataset[0])
print("Labels:", dataset[1])

print("Converting to token spans (GLiNER) format:")
dataset_gliner_f = hf_dataset_to_gliner_format(dataset[0]["test"], dataset[1])

print(dataset_gliner_f[0])

Loading and merging from HF:
--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192
Dataset: {'test': Dataset({
    features: ['id', 'text', 'tokens', 'ner_tags'],
    num_rows: 192
})}
Labels: ['O', 'B-Asset', 'I-Asset', 'B-Body Part', 'I-Body Part', 'B-Body of Water', 'I-Body of Water', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease', 'B-Ecosystem', 'I-Ecosystem', 'B-Energy Source', 'I-Energy Source', 'B-Field of Study', 'I-Field of Study', 'B-Geographical Feature', 'I-Geographical Feature', 'B-Intellectual Artefact', 'I-Intellectual Artefact', 'B-Location', 'I-Location', 'B-Mathematical Expression', 'I-Mathematical Expression', 'B-Measuring Device', 'I-Measuring Device', 'B-Meteorological Phenomenon', 'I-Meteorological Phenomenon', 'B-Method', 'I-Method', 'B-Natural Disaster', 'I-Natural Disaster', 'B-Natural Phenomenon', 'I-Natural Phenomenon', 'B-Organism', 'I-Organism', 'B-Organization',

In [ ]:
from dataset_processing import (
    cwed4eta_process_json_file,
    convert_to_token_spans,
    CLIRENER_LABELS_V1,
    process_directory_of_json_files,
)

ANNOTATOR_DIR = "/home/p0l3/RAD/DROP/CLIRENER/ANNOTATORS/"

data_gold_merged = convert_to_token_spans(cwed4eta_process_json_file(ANNOTATOR_DIR + "CONSENSUS/" + "6326.json", [1]))


In [36]:
data_gold_merged

[{'id': '10167-54',
  'text': 'Excluding time points when IM PB was administered within 7 days prior to sampling (n = 11), peak samples collected <4 h after dosing (n = 3), and trough samples collected >28 h after dosing (n = 14), thirty-five time points were trough samples collected 24 h after dosing.',
  'tokenized_text': ['Excluding',
   'time',
   'points',
   'when',
   'IM',
   'PB',
   'was',
   'administered',
   'within',
   '7',
   'days',
   'prior',
   'to',
   'sampling',
   '(',
   'n',
   '=',
   '11',
   ')',
   ',',
   'peak',
   'samples',
   'collected',
   '<',
   '4',
   'h',
   'after',
   'dosing',
   '(',
   'n',
   '=',
   '3',
   ')',
   ',',
   'and',
   'trough',
   'samples',
   'collected',
   '>',
   '28',
   'h',
   'after',
   'dosing',
   '(',
   'n',
   '=',
   '14',
   ')',
   ',',
   'thirty-five',
   'time',
   'points',
   'were',
   'trough',
   'samples',
   'collected',
   '24',
   'h',
   'after',
   'dosing',
   '.'],
  'ner': [[1, 2, 'Time P